In [1]:
import os, subprocess, sys, shutil

def run(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise SystemExit(f'{what} failed (exit {r.returncode})')
    print(f'{what} OK')

ON_KAGGLE = os.path.isdir('/kaggle/working')
PERSIST = '/kaggle/working/turnwave-ckpt' if ON_KAGGLE else '/content/drive/MyDrive/turnwave'
REPO = '/kaggle/working/turnwave' if ON_KAGGLE else '/content/turnwave'
if not ON_KAGGLE:
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(PERSIST, exist_ok=True)
if not os.path.isdir(REPO):
    run(['git','clone','-q','https://github.com/Nikhils-G/turnwave.git', REPO], 'clone')
%cd {REPO}
subprocess.run(['git','checkout','--','.']); subprocess.run(['git','pull','-q'])
pip = [sys.executable,'-m','pip','install','-q']
run(pip + ['--no-deps','-e','.'], 'turnwave')
run(pip + ['sentencepiece','soundfile','datasets','onnx','onnxruntime','onnxscript','faster-whisper'], 'deps')
import torch
assert torch.cuda.is_available(), 'No GPU — set the accelerator'
print('GPU', torch.cuda.get_device_name(0), '| persist ->', PERSIST)


clone OK
/kaggle/working/turnwave
turnwave OK
deps OK
GPU Tesla T4 | persist -> /kaggle/working/turnwave-ckpt


In [2]:

BASE = 'https://github.com/Nikhils-G/turnwave/releases/download/models-v1'
for d in ('checkpoints/text_eot','checkpoints/audio_eot_v2','checkpoints/tokenizer'):
    os.makedirs(d, exist_ok=True)
!wget -q {BASE}/text_eot.best.pt     -O checkpoints/text_eot/best.pt
!wget -q {BASE}/audio_eot_v2.best.pt -O checkpoints/audio_eot_v2/best.pt
!wget -q {BASE}/spm.model            -O checkpoints/tokenizer/spm.model
assert os.path.getsize('checkpoints/text_eot/best.pt') == 27686397, 'text ckpt truncated'
assert os.path.getsize('checkpoints/audio_eot_v2/best.pt') == 14004001, 'audio ckpt truncated'
print('checkpoints ready')


checkpoints ready


In [3]:
!python scripts/transcribe_clips.py --config eng --measure 200

whisper base.en on cuda (float16)
README.md: 100%|███████████████████████████████| 311/311 [00:00<00:00, 1.43MB/s]
Repo card metadata block was not found. Setting CardData to empty.
Resolving data files: 100%|██████████████████| 77/77 [00:00<00:00, 29534.65it/s]
Traceback (most recent call last):
  File "/kaggle/working/turnwave/scripts/transcribe_clips.py", line 164, in <module>
    main()
  File "/kaggle/working/turnwave/scripts/transcribe_clips.py", line 112, in main
    ds = (load_dataset(args.dataset, args.config, split=args.split, streaming=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1698, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1363, in load_dataset_builder
    builder_instance: DatasetBuilder = builder_cls(
                   

In [4]:
!python scripts/transcribe_clips.py --split eng --measure 200

whisper base.en on cuda (float16)
Repo card metadata block was not found. Setting CardData to empty.
Resolving data files: 100%|██████████████████| 77/77 [00:00<00:00, 29942.65it/s]
  200 clips  (266/min, 0 empty)
done: 200 transcribed in 0.8 min = 265 clips/min (0 empty)
  projected 20,000 clips: 75 min
  projected 40,000 clips: 151 min
  projected 65,000 clips: 245 min


In [5]:
!python scripts/transcribe_clips.py --split eng --max-clips 40000 \
    --out {PERSIST}/transcripts_train.jsonl

whisper base.en on cuda (float16)
Repo card metadata block was not found. Setting CardData to empty.
Resolving data files: 100%|██████████████████| 77/77 [00:00<00:00, 29137.62it/s]
  200 clips  (259/min, 0 empty)
  400 clips  (378/min, 0 empty)
  600 clips  (457/min, 0 empty)
  800 clips  (507/min, 0 empty)
  1200 clips  (462/min, 0 empty)
  1400 clips  (492/min, 0 empty)
  1600 clips  (516/min, 1 empty)
  1800 clips  (461/min, 1 empty)
  2000 clips  (481/min, 1 empty)
  2200 clips  (500/min, 1 empty)
  2400 clips  (517/min, 1 empty)
  2600 clips  (525/min, 1 empty)
  2800 clips  (539/min, 1 empty)
  3000 clips  (548/min, 1 empty)
  3200 clips  (558/min, 1 empty)
  3400 clips  (569/min, 1 empty)
  3600 clips  (573/min, 1 empty)
  3800 clips  (582/min, 1 empty)
  4000 clips  (591/min, 1 empty)
  4200 clips  (599/min, 1 empty)
  4400 clips  (600/min, 1 empty)
  4600 clips  (607/min, 1 empty)
  4800 clips  (613/min, 1 empty)
  5000 clips  (619/min, 1 empty)
  5200 clips  (619/min, 1 empt

In [6]:
!python scripts/transcribe_clips.py \
    --dataset pipecat-ai/smart-turn-data-v3.2-test --languages eng \
    --max-clips 9000 --out {PERSIST}/transcripts_eval.jsonl

whisper base.en on cuda (float16)
README.md: 3.25kB [00:00, 8.56MB/s]
  200 clips  (690/min, 0 empty)
  400 clips  (735/min, 0 empty)
  600 clips  (767/min, 0 empty)
  800 clips  (765/min, 0 empty)
  1000 clips  (764/min, 0 empty)
  1200 clips  (772/min, 0 empty)
  1400 clips  (777/min, 0 empty)
  1600 clips  (774/min, 0 empty)
  1800 clips  (774/min, 0 empty)
  2000 clips  (783/min, 0 empty)
  2200 clips  (788/min, 1 empty)
  2400 clips  (784/min, 1 empty)
  2600 clips  (780/min, 2 empty)
  2800 clips  (784/min, 2 empty)
  3000 clips  (787/min, 2 empty)
  3200 clips  (786/min, 2 empty)
  3400 clips  (785/min, 2 empty)
  3600 clips  (788/min, 3 empty)
  3800 clips  (787/min, 3 empty)
  4000 clips  (783/min, 3 empty)
  4200 clips  (782/min, 3 empty)
  4400 clips  (783/min, 3 empty)
  4600 clips  (785/min, 3 empty)
  4800 clips  (778/min, 3 empty)
  5000 clips  (780/min, 4 empty)
  5200 clips  (780/min, 5 empty)
  5400 clips  (783/min, 5 empty)
  5600 clips  (780/min, 5 empty)
  5800 cli

In [7]:
!cat {PERSIST}/transcripts_train.jsonl {PERSIST}/transcripts_eval.jsonl > /tmp/transcripts.jsonl
!python scripts/build_audio_dataset.py --source smart-turn --languages eng \
    --splits validation test --max-eval-examples 4000 \
    --transcripts /tmp/transcripts.jsonl --out data/audio_p6
!python scripts/build_audio_dataset.py --source smart-turn \
    --dataset giangndm/smart-turn-data-v3.1-en-vi --hf-split eng --splits train \
    --max-examples 40000 --transcripts /tmp/transcripts.jsonl --out data/audio_p6


transcripts: 47,820 clips covered
test      : 100%|███████████████████████████| 4000/4000 [01:17<00:00, 51.63ex/s]
validation     4,000 examples from  4,000 rows (1,989 complete / 2,011 mid-turn, 49.7% positive)
test           4,000 examples from  4,000 rows (1,995 complete / 2,005 mid-turn, 49.9% positive)
manifest: data/audio_p6/manifest.json
Fatal Python error: PyGILState_Release: thread state 0x7dc4adc46890 must be current when releasing
Python runtime state: finalizing (tstate=0x0000000000b8a5b0)

Thread 0x00007dc6b1f36480 (most recent call first):
  <no Python frame>

Extension modules: numpy._core._multiarray_umath, numpy._core._multiarray_tests, numpy.linalg._umath_linalg, _cffi_backend, torch._C, torch._C._dynamo.autograd_compiler, torch._C._dynamo.eval_frame, torch._C._dynamo.guards, torch._C._dynamo.utils, torch._C._fft, torch._C._linalg, torch._C._nested, torch._C._nn, torch._C._sparse, torch._C._special, zstandard.backend_c, pyarrow.lib, numpy.random._common, numpy.random.

In [8]:
!git -C /kaggle/working/turnwave pull -q && echo pulled

pulled


In [9]:
!cat {PERSIST}/transcripts_train.jsonl {PERSIST}/transcripts_eval.jsonl > /tmp/transcripts.jsonl
!python scripts/build_audio_dataset.py --source smart-turn --languages eng \
    --splits validation test --max-eval-examples 4000 \
    --transcripts /tmp/transcripts.jsonl --out data/audio_p6
!python scripts/build_audio_dataset.py --source smart-turn \
    --dataset giangndm/smart-turn-data-v3.1-en-vi --hf-split eng --splits train \
    --max-examples 40000 --transcripts /tmp/transcripts.jsonl --out data/audio_p6

transcripts: 47,820 clips covered
test      :  96%|█████████████████████████▊ | 3820/4000 [02:09<00:06, 29.59ex/s]
validation     4,000 examples from  4,000 rows (1,989 complete / 2,011 mid-turn, 49.7% positive)
test           3,820 examples from  3,820 rows (1,844 complete / 1,976 mid-turn, 48.3% positive)
manifest: data/audio_p6/manifest.json
transcripts: 47,820 clips covered
Repo card metadata block was not found. Setting CardData to empty.
train     : 100%|████████████████████████| 40000/40000 [06:09<00:00, 108.38ex/s]
train         40,000 examples from 40,000 rows (19,827 complete / 20,173 mid-turn, 49.6% positive)
manifest: data/audio_p6/manifest.json


In [10]:
  import json
  ids = {}
  for sp in ('train','validation','test'):
      rows = [json.loads(l) for l in open(f'data/audio_p6/{sp}.jsonl')]
      ids[sp] = {r['id'] for r in rows}
      cov = sum(1 for r in rows if r['text']) / len(rows)
      print(f'{sp:11s} {len(rows):6d} rows | text coverage {cov:.1%}')
  assert not ids['train'] & ids['validation'], 'train/val LEAK'
  assert not ids['train'] & ids['test'], 'train/test LEAK'
  assert not ids['validation'] & ids['test'], 'val/test LEAK'
  print('splits disjoint — safe to train')

train        40000 rows | text coverage 100.0%
validation    4000 rows | text coverage 99.9%
test          3820 rows | text coverage 99.8%
splits disjoint — safe to train


In [11]:
!python -m turnwave.train --task fusion --cache data/audio_p6 \
    --tokenizer checkpoints/tokenizer/spm.model \
    --text-ckpt checkpoints/text_eot/best.pt \
    --audio-ckpt checkpoints/audio_eot_v2/best.pt \
    --out {PERSIST}/fusion_eot_v2 --steps 2500 --batch-size 128 --lr 1e-3 \
    --num-workers 2 --resume

device=cuda task=fusion params=10.54M (trainable 0.13M) train=40000 val=4000
no checkpoint at /kaggle/working/turnwave-ckpt/fusion_eot_v2/last.pt; starting fresh
step    250  lr 8.33e-04  train 0.3036  val 0.2989  acc 0.876  f1 0.885  ap 0.938
step    500  lr 9.80e-04  train 0.2400  val 0.2662  acc 0.888  f1 0.892  ap 0.939
step    750  lr 9.02e-04  train 0.2234  val 0.2630  acc 0.888  f1 0.890  ap 0.942
step   1000  lr 7.73e-04  train 0.2257  val 0.2723  acc 0.886  f1 0.892  ap 0.945
step   1250  lr 6.11e-04  train 0.2221  val 0.2759  acc 0.885  f1 0.891  ap 0.944
step   1500  lr 4.35e-04  train 0.2164  val 0.2540  acc 0.894  f1 0.897  ap 0.946
step   1750  lr 2.68e-04  train 0.2168  val 0.2552  acc 0.893  f1 0.895  ap 0.946
step   2000  lr 1.31e-04  train 0.2152  val 0.2519  acc 0.894  f1 0.897  ap 0.948
step   2250  lr 4.15e-05  train 0.2102  val 0.2538  acc 0.895  f1 0.897  ap 0.948
step   2500  lr 1.00e-05  train 0.2097  val 0.2527  acc 0.893  f1 0.897  ap 0.948
done. best val AP 

In [12]:
!python -m turnwave.ablate --cache data/audio_p6 \
    --tokenizer checkpoints/tokenizer/spm.model \
    --text-ckpt checkpoints/text_eot/best.pt \
    --audio-ckpt checkpoints/audio_eot_v2/best.pt \
    --fusion-ckpt {PERSIST}/fusion_eot_v2/best.pt \
    --split test --device cuda --out {PERSIST}/ablation_p6.json
!python -m turnwave.export --ckpt {PERSIST}/fusion_eot_v2/best.pt --out-dir {PERSIST}/onnx
!python scripts/plot_training.py {PERSIST}/fusion_eot_v2/log.csv --out {PERSIST}/fusion_v2_curves.png
!zip -qr /kaggle/working/phase6.zip {PERSIST}
print('>>> download phase6.zip from the Output panel BEFORE cell 10 <<<')


data/audio_p6/test: 3,820 examples (1,844 turn-final / 1,976 mid-turn)

                             acc    prec  recall      f1      ap
majority class             0.517   0.000   0.000   0.000   0.492
cue-word heuristic         0.712   0.628   0.985   0.767   0.629
text only                  0.704   0.636   0.905   0.747   0.665
audio only                 0.843   0.831   0.845   0.838   0.905
fused (text + audio)       0.897   0.855   0.946   0.899   0.959

fusion helps: AP 0.959 vs 0.905 (audio only), delta +0.054
wrote /kaggle/working/turnwave-ckpt/ablation_p6.json
/kaggle/working/turnwave/turnwave/export.py:84: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.h

In [13]:
!ls -lh /kaggle/working/phase6.zip


-rw-r--r-- 1 root root 125M Sep  1 19:04 /kaggle/working/phase6.zip
